## 

# 2 Transformer 核心模块
## 2.1 一批 Token 是如何流过模型的？
![](/img/pytorch/transformer.png)
可以先把整个语言模型看成这样一条流水线：
```
Token IDs [B, S]
   ↓
Embedding
   ↓
Hidden States [B, S, d_model]
   ↓
N 个 Transformer Block
   ↓
Final Norm
   ↓
LM Head
   ↓
Logits [B, S, vocab_size]
   ↓
Softmax（训练时通常不显式写）
   ↓
每个位置预测“下一个 Token”的概率分布
```
- B：Batch Size，一次处理多少条样本
- S：Sequence Length，每条样本有多少个 Token
- d_model：模型主隐藏维度
- vocab_size：词表大小
- h：注意力头数
- d_head = d_model / h：每个注意力头的维度

### 2.1.1 输入阶段
模型最开始拿到的，并不是字符串，而是 Tokenizer 编码后的整数张量：`token_ids.shape == [B, S]`
```
[
  [332, 545, 612],
  [866, 959, 1402]
]
```
> 上面有2条样本，Batchsize=2；每条样本长度为3，所以SequenceLength=3

这些整数本身没有“数学语义”，Token ID 是离散符号，不是连续特征。
而神经网络的核心操作是向量加法、矩阵乘法、点积、非线性变换，所以模型必须先把离散的 ID 变成连续向量。这就是 Embedding 层存在的原因。

### 2.1.2 Embedding
Embedding 的本质可以概括为一个巨大的 **“查找表”**。
- 输入：`[B, S]`
- 输出：`[B, S, d_model]`
这里BPE Tokenizer输出的token ids每一个都对应一行参数，这个参数就是长度为`d_model`的浮点向量。
给定一个token id，就是去表里查“取第几行”。这些特征向量就可以去做**线性变换、注意力计算与非线性加工**。
**训练完成后，语义接近的token往往会在向量空间中更接近。**当然训练前肯定不是。
> 比如一个token id=545，输出就是一个d_model维度的向量比如说是4096

### 2.1.3 Transformer Block
一个Block通常由两个核心子模块组成：
1. self-attention：让不同位置间交换信息（看上下文）
2. FFN：让每个位置加工自己的特征（消化自己的信息）
#### 自注意力
在自注意力力，每个序列中的位置都在问：对我当前这个位置来说，前面哪些位置最重要？
模型会对每个位置的向量做三次线性投影，得到三种角色：
- Q（Query）：我想找什么
- K（Key）：我这里提供什么索引特征
- V（Value）：我这里真正携带的信息内容

输入`X.shape == [B, S, d_model]`，得到`Q=XW_Q、K=XW_K、V=XW_K`，形状都是`[B, S, d_model]`，都是**完全独立随机初始化**的。
对于某个位置t，我们拿它的Q_t去和所有位置的K_t做点积，点积越大，说明当前位置更应该关注那个位置。
最后得到**分数矩阵**：
$$scores = \frac{QK^\top}{\sqrt{d}}$$
> 这里的d是单头维度，即d_model/num_head，目的是为了数值稳定，避免点积随着维度变大而过大

得到**分数**后，对分数做Softmax得到0~1的概率，得到一组注意力权重，确定「我该重点关注谁」

用这组权重，也就是**相关度分数**去加权求和所有Value：
$$\text{Attention}(Q,K,V) = \text{softmax}\left( \frac{QK^\top}{\sqrt{d_k}} \right)V$$
最终才会融合真正的**特征信息**。由此一来，当前位置从整个上下文中，按需取回一份“上下文摘要”。

#### 多头注意力
单头注意力可以理解为用一种方式看上下文；
多头注意力则是用多种不同视角并行看上下文。例如某些头可能更擅长：
- 关注最近的词
- 关注主谓关系
- 关注标点或边界
- 关注长距离依赖
- 关注格式模式
因此，多头机制允许模型在多个子空间中并行建模。
> 为了保证这一点，所有小头的权重，都是 **独立随机初始化**


首先我们有一个`d_model = num_head * d_head`。把`Q/K/V`最后一维拆成多个头：`[B, S, num_head, d_head]`；
之后再**转置**成`[B, num_head, S, d_head]`，因为计算注意力时，`Q/K/V`**按head维度并行更方便**。
`Q × K_T` 后得到的**分数矩阵**形状是`[B, num_head, S, S] = [B, h, S, d_head] x [B, h, d_head, S]`（只是后面矩阵相乘(S, d_head) × (d_head, S) = (S, S)）
也就是每个批次、每个头、第 i 个词 对 所有 S 个词的相关性打分
- 第一个 S：当前词
- 第二个 S：所有要对比的词

分数矩阵左乘以矩阵V：`[B, num_head, S, S] x [B, num_head, S, d_head] = [B, num_head, S, d_head]`
也就是每个单头的注意力权重就是这个形状，再转置回去`[B, S, num_head, d_head]`并reshape成`[B, S, d_model]`
> 最后通常还会再经过一个输出投影层 W_O，输出 shape 保持不变

#### 因果编码
语言模型的训练目标是 **用前面的 token 去预测后面的 token**，所以第 t 个位置在计算时，只能看自己和自己之前的位置，不能偷看未来。否则训练就作弊了。

在注意力分数矩阵`[S, S]`上：
- 主对角线以上：小Q对大K，表示未来位置，不准看
- 主对角线及以下：OK

| 矩阵维度 | 名称 | 对应 | 含义 |
|---------|------|------|------|
| **行 (i)** | Query 位置 | Q | **当前正在看的词**（我是谁） |
| **列 (j)** | Key 位置 | K | **我能看的词**（看谁） |

```

          j=0   j=1   j=2   j=3
        +-----------------------+
i=0(词1)| ①     ②     ③     ④  |  ← 词1居然能看到词2/3/4（未来！违法）
i=1(词2)| ⑤     ⑥     ⑦     ⑧  |  ← 词2能看到词3/4（未来！违法）
i=2(词3)| ⑨     ⑩    ⑪    ⑫  |  ← 词3能看到词4（未来！违法）
i=3(词4)| ⑬    ⑭    ⑮    ⑯  |  ← 词4看全部（合法）
        +-----------------------+


          j=0   j=1      j=2        j=3
        +-----------------------------------+
i=0(词1)| ①    -∞      -∞        -∞     |  只看自己
i=1(词2)| ⑤     ⑥      -∞        -∞     |  看自己+前一个
i=2(词3)| ⑨     ⑩      ⑪        -∞     |  看前两个+自己
i=3(词4)| ⑬    ⑭      ⑮        ⑯     |  看全部
        +-----------------------------------+
```

总的来说，因果编码的目的就是为了满足 **自回归语言模型的因果约束**。
#### RoPE: Rotary Position Embedding
如果没有任何位置机制，那么自注意力只会知道来了一堆token向量，却不知道谁在前谁在后，或两个token相隔多远。
RoPE的思路是在Q和K上施加与位置相关的**旋转变换**。
**传统绝对位置编码（正弦/可学习PE）**：
$$X_{pos} = X + P$$
然后用这个带位置的 $X$ 去算：$Q=XW_q,\,K=XW_k,\,V=XW_v$
然而绝对位置编码它有3个致命硬伤
1. **位置污染了V（完全没必要）**
注意力逻辑本来是：
- $Q,K$：负责算「谁和谁有关系」→ 需要位置
- $V$：负责存「纯内容特征」→ **根本不需要位置**
直接加输入 = 强行把位置塞进了 $V$，多余、干扰语义。
2. 只能学**绝对位置**，不懂「相对距离」
比如：
- 句子第1、2个词（间隔1）
- 句子第100、101个词（间隔1）
老式PE：两个间隔1的词，相似度算出来**不一样**（因为绝对位置不同）
但人类语言里：**相邻词的亲密关系，和它在句子开头/结尾无关**！
3. 长文本外推拉胯
训练只见过短位置（比如≤2048），推理超长文本（4096/8192）时，陌生的绝对位置直接崩，模型乱关联。

与此同时，RoPE 核心操作不改输入、不加向量；只在注意力打分前，给每个位置的 $Q,K$ 做**角度旋转**：
- 位置 $m$ 的 $Q_m$ 转一个角度
- 位置 $n$ 的 $K_n$ 转一个角度
然后再算：$\boldsymbol{Q_m \cdot K_n}$
1. **只影响打分，不污染内容**
只动 $Q,K$（相似度计算环节），**完全不动V**！
位置信息只用来判断「该关注谁」，不篡改原本的语义特征，分工极致干净。
2. 天然自带**相对位置感知**
数学上可以推出来：
$Q_m \cdot K_n$ 的结果，**只和两个token的间隔 $(m-n)$ 有关**，和它们在句子里的绝对位置无关！
不管你在开头、中间、结尾，只要俩词隔3个位，关联强度永远一致——贴合人类语言逻辑。
3. 天生支持长文本外推
旋转是连续三角函数，没固定位置上限；LLaMA、Qwen全系大模型标配RoPE，就是因为能无缝撑超长上下文。

> 具体的代码见后面

#### FFN / SwiGLU
Attention 之后，每个位置已经把上下文信息读进来了，接下来模型还需要对每个位置自己的表示做进一步加工，这就是 **前馈神经网络** 的作用。
其基本操作就是升维、激活再降维：`Linear(d_model->d_ff) ——> 激活函数 ——> Linear(d_ff->d_model)`
对应过来的shape变化就是`[B, S, d_model] ——> [B, S, d_ff] ——> [B, S, d_model]`
$$FFN(x) = ReLU(x\cdot W_{up}) \cdot W_{down}$$

更大的中间维度，意味着：
- 更强的表达能力
- 更多的特征组合空间
- 更丰富的非线性加工能力

当然，传统ReLU好坏特征全混在一起，没用的噪音也留着，有点难绷，所以这里我们都是用门控机制（话说NIPS2025就有一篇说Qwen的门控注意力的？）
像这里绝大多数开源大模型用到的都是`SwiGLU = Swish + GLU`
- $$\sigma(x) = \frac{1}{1+e^{-x}}$$ 即 Sigmoid 函数
- $$\text{Swish}(x) = x \cdot \sigma(x)$$，如此以来 Swish 全程连续可导；负数不直接置 0，是平滑衰减

接着复习一下 GLU：
输入 $x$，拆分两个独立线性投影：
$$
\begin{align}
A &= x W_A \\
B &= x W_B
\end{align}
$$
$W_A,W_B$：可学习权重矩阵
$\odot$：**逐元素相乘（Hadamard Product）**
则 $$\text{GLU}(x) = \sigma(A) \odot B$$
- $\sigma(A)$：生成 0~1 的门控权重
- $B$：原始特征通路
- 相乘：门控过滤特征

最后 SwiGLU 顾名思义，就是把 GLU 的 Sigmoid 门换成 Swish：
$$
\text{SwiGLU}_\text{core}(x) = \text{Swish}(xW_A) \odot xW_B
$$

### 2.1.4 Block 输出
经过多个 Transformer Block 后，输出 shape `[B, S, d_model]`。但是我一直有一个疑问：**凭啥 Transformer Block 叠几层，向量就「语义更高级」？**
它全程无人类语义标注，只有一条数学任务：**给前 t 个词，猜第 t+1 个词（预测下一个 token）**
答案是——这是训练出来的，梯度惩罚。
比如说“苹果” 的向量，必须能精准算出下一个大概率是 “手机 / 水果”。
光靠原始词向量（刚查表出来的）根本做不到，因为单个 token 向量是孤立的，不知道上下文
那只能靠 Self-Attention 把整句话所有词的向量，加权揉在一起
- “我 吃 苹果” → 让「苹果」偷偷吸收「吃」的动作信息
- “我 用 苹果” → 让「苹果」吸收「用」的工具信息

就像多层卷积一样，叠多层 Transformer Block后：
- 第一层：学局部搭配（吃 + 苹果）
- 第二层：学短句逻辑（我吃苹果→常识）
- 第三层：学长距离关联（前文指代、逻辑因果）

既然聊到梯度了，就不得不提 **叠几十上百层** 的深度神经网络里非常可怕的现象：
- 某一层数值偏大 → 放大传到下一层 → 越传越大 → 梯度爆炸
- 某一层数值全挤很小 → 梯度趋近 0 → 完全学不动

#### LayerNorm
第一个归一化的方法是 LayerNorm，也就是对单个样本，单个向量，在最后一维（特征维）算这一条向量的均值与方差，将其强行标准化拉到均值0、方差1：
$$\hat x_i = \frac{x_i - \mu}{\sqrt{\sigma^2+\epsilon}}$$
> epsilon是一个超级小的正数比如 10^-6，防止方差为0
再用可学习缩放偏移：$y_i = \gamma \hat x_i + \beta$
- gamma 是模型自己训练更新的参数，不是固定数，把归一化后方差 = 1 的向量，按需放大（重要特征） / 缩小（无用特征）

不过这样因为算均值有减法，计算量方面比较大。特地摘出来说的原因就是下面的RMSNorm不算减法。

#### RMSNorm
RMSNorm（Root Mean Square Normalization）实验证明，在 Transformer 里，仅用均方根（RMS）做缩放，不减去均值，效果几乎不降，但计算更快、更省显存。
$$ \text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^d x_i^2 + \epsilon} $$

$$ \hat x_i = \frac{x_i}{\text{RMS}(x)} \cdot \gamma $$

- **没有减去均值** $\mu$，只除以 RMS 值。
- $\gamma$ 同样是可学习的缩放参数。
- 输出均值不再强制为 0，但方差被约束在 $\gamma^2$ 附近。

| 步骤 | LayerNorm | RMSNorm |
|------|-----------|---------|
| 计算均值 | 需要 | 跳过 |
| 计算方差 | 需要 | 用均方根替代 |
| 归一化 | 减均值、除标准差 | 仅除 RMS |
| 可学习参数 | $\gamma, \beta$ | $\gamma$（无偏置） |
| 计算量 | 高 | 低约 20%～30% |

> **为什么可以丢掉均值信息？**
> 因为 Transformer Block 内部有残差连接和全连接层，后者本身就包含偏置项（bias），可以隐式地学习均值偏移。去掉显式的均值中心化，相当于把这份工作交给了后续的线性层，反而释放了归一化层的算力瓶颈。

光有归一化还不够，深层网络还有一个更根本的问题：**梯度消失**。想象一下，第 50 层的梯度要一路传回第 1 层，中间经过几十次矩阵乘法和激活函数。如果每层的梯度略小于 1（比如 0.9），$0.9^{50} \approx 0.005$，梯度几乎没了；如果略大于 1（比如 1.1），$1.1^{50} \approx 117$，直接爆炸。

$$ \text{Output} = \text{LayerNorm}(X + \text{Sublayer}(X)) $$

```python
# 伪代码展示一个 Transformer Block 内部的两个残差支路
def transformer_block(x):
    # 1. 自注意力 + 残差
    x = x + self_attention(norm1(x))   # norm 通常是 pre-norm
    # 2. FFN + 残差
    x = x + ffn(norm2(x))
    return x
```

- **Pre-Norm**（现代主流）：先 Norm，再做子层变换，最后加回原始输入。Pre-Norm 的优势在于梯度流动更顺畅，训练更稳定，因此几乎所有开源大模型都采用 Pre-Norm + 残差结构。
- **Post-Norm**（早期 Transformer）：先子层变换，再加回输入，最后 Norm。
残差连接的价值不仅在于梯度，还在于 **信息保真**：
- 即使某个子层完全学废了（输出全 0 或噪声），残差通路仍能把原始输入无损传到下一层。
- 这让模型可以堆叠上百层而不崩。


当输入经过了 $N$ 个 Transformer Block 的反复加工后，输出的 hidden states 形状依然是 `[B, S, d_model]`。此时每个位置的向量已经深度融合了上下文的语义信息。

但不同层的输出分布可能存在微小漂移，因此在送入最后的 **语言模型头（LM Head）** 之前，需要再做一次 **Final Norm**（通常是 RMSNorm）。
作用有三：
1. **稳定输出分布**：让不同位置的向量处于同一个数值尺度上，防止某些位置激活值过大或过小。
2. **解耦训练**：让最后一个线性层接收到的输入更规整，优化更平稳。
3. **极低成本**：就一层归一化，计算量可忽略。

#### LM Head
Final Norm 后的 hidden states 形状为 `[B, S, d_model]`，而我们的最终目标是预测下一个 token，所以需要把它映射回词表空间：`[B, S, vocab_size]`。

这个映射层就是 **LM Head**，通常就是一个 **线性层（无偏置）**：

$$ \text{logits} = X_{\text{final}} W_{\text{head}}^\top $$

- $X_{\text{final}} \in \mathbb{R}^{B \times S \times d_{\text{model}}}$
- $W_{\text{head}} \in \mathbb{R}^{\text{vocab\_size} \times d_{\text{model}}}$
- 输出 $\text{logits} \in \mathbb{R}^{B \times S \times \text{vocab\_size}}$

> **权重共享**：在许多模型中，$W_{\text{head}}$ 与输入 Embedding 层的权重矩阵是 **同一份参数**（转置关系）。这叫 **Weight Tying**，能大幅减少参数量（词表 × d_model 往往占模型总参数的 30% 以上），同时让输入和输出空间对齐，提升泛化性。

此时，对于序列中的每一个位置 $t$，我们都得到了一个长度为 `vocab_size` 的向量 logits，每个元素代表模型认为 **下一个 token 是词表中对应 ID 的原始分数**（未归一化）。

#### Softmax
拿到 logits 后，最后一步就是把分数转换为概率，并计算与真实下一个 token 的差距。这就需要用到超级经典的 **Softmax 函数**：

$$ P(y_i) = \frac{e^{z_i}}{\sum_{j=1}^{\text{vocab\_size}} e^{z_j}} $$

其中 $z_i$ 是 logits 向量中第 $i$ 个元素。结果满足 $\sum P = 1$，且每个 $P \in (0,1)$。

> **训练时的技巧**：由于 vocab_size 通常极大（5 万～ 32 万+），完整计算 Softmax 分母的求和非常昂贵。实际训练中普遍使用 **交叉熵损失 + 稀疏 Softmax**，只计算真实标签对应位置的梯度，分母用采样或近似（如 Sampled Softmax 采样一部分负类；Noise Contrastive Estimation；**Fused Cross-Entropy** 算子硬加速）。

**交叉熵损失**（Cross-Entropy Loss）直接衡量预测分布与真实分布的差异：

$$
H(p, q) = -\sum_{i=1}^{V} p_i \cdot \log(q_i)
$$

- \(p_i\)：**真实分布**在第 i 个 token 上的值（0 或 1）
- \(q_i\)：**模型预测分布**在第 i 个 token 上的概率（Softmax 输出）
- \(V\)：词表大小 vocab_size
- 前面加负号：因为 log(q_i) 是负数，负负得正，让损失变成正数，方便优化

假设词表大小 vocab_size = 5， tokens 是 `[我，爱，编，码，EOS]`；现在真实的下一个 token 是`爱`，它对应的真实分布就是：
```
[0, 1, 0, 0, 0]
```
只有正确 token 位置是 1、其他全是 0 的独热分布。接下来经过 LM Head 到 Softmax 之后，假设得到的概率是：
```
[0.1, 0.7, 0.1, 0.05, 0.05]
```
因为**每个位置只有一个正确 token**，真实分布 `p_i` 只有一个位置是 1，其余全是 0。

于是**求和直接消失**，公式瞬间简化成：
$$
\mathcal{L} = -\log(q_{\text{true}})
$$

接上上面的例子：
- 正确 token：爱
- 模型给它的概率：\(q_{\text{true}} = 0.7\)

计算损失：
$$
\mathcal{L} = -\log(0.7) \approx 0.357
$$

如果模型更准，比如概率升到 0.95：
$$
\mathcal{L} = -\log(0.95) \approx 0.051
$$

如果模型很烂，只给 0.1：
$$
\mathcal{L} = -\log(0.1) \approx 2.30
$$
- 正确 token 概率 → 越高，损失 → 越小
- 正确 token 概率 → 越低，损失 → 爆炸变大
- 反向传播时，梯度会沿着 LM Head → Final Norm → 所有 Transformer Blocks → Embedding 一路回传，更新全部参数。

$$
\mathcal{L} = - \sum_{t=1}^{S} \log P(y_t^* \mid x_{<t})
$$

现在完全能看懂了：
- \(t\)：序列里的每一个位置（比如一句话有 512 个 token，就有 512 个位置）
- \(S\)：序列长度
- \(y_t^*\)：第 t 个位置的**真实下一个 token**
- 对每个位置都算一次 `-log(概率)`，然后全部加起来，就是**整句话的总损失**

> 另外 `(1 - q_true)²`的 MSE 不太行，因为它对接近 0/1 的概率梯度极小，交叉熵就没有这个问题，在概率低时梯度极大，并且本来他就是天生用来衡量两个概率分布差异的标准工具。

训练时我们并行计算所有位置的 logits，因为 ground truth 序列已知。但 **推理（生成）时是串行的**：

1. 输入一个初始 prompt（比如 `"你好，"`），模型输出下一个 token 的概率分布。
2. 按一定策略（贪心、温度采样、Top-p 等）选出一个 token。
3. 将该 token 拼接到序列末尾，形成新的输入。
4. 重复 1~3，直到输出结束符（EOS）或达到最大长度。

此时为了加速，会使用 **KV Cache** 缓存已计算的 Key 和 Value，避免重复计算历史 token 的注意力（后面章节会详述）。

#### 完整的前向流
```
Token IDs        [B, S]
    ↓  Embedding
Hidden           [B, S, d_model]
    ↓  × N Blocks (每个内含 Attention + FFN + 残差 + RMSNorm)
Hidden           [B, S, d_model]
    ↓  Final RMSNorm
Hidden           [B, S, d_model]
    ↓  LM Head
Logits           [B, S, vocab_size]
    ↓  Softmax (训练隐式，推理显式)
Probs            [B, S, vocab_size]
    ↓  Argmax / Sampling
Next Token IDs   [B, 1]  (推理阶段追加)
```

## 2.2 Embedding
还记得模型拿到的第一手数据长什么样吗？
```bash
token_ids = [
    [332, 545, 612],
    [866, 959, 1402]
]
# shape: [B=2, S=3]
```
这些整数本身只是字典里的编号。545 和 866 之间的数学关系，并不是“苹果”和“香蕉”的语义距离——它们只是碰巧被分配到这个 ID 而已。如果直接把这些整数喂给矩阵乘法，模型会把 ID 的大小当作一种数值特征，这是完全错误且灾难性的。
因此，Embedding 层的使命非常纯粹，就是将离散的 Token ID 映射为连续的高维向量，让每个词变成一个可以计算、可以比较、可以被神经网络加工的点。
Embedding 层实现方式也极其直白：维护一个巨大的参数矩阵查找表：
```
weight.shape == [num_embeddings, embedding_dim]
```
其中：
- num_embeddings = vocab_size
- embedding_dim = d_model


如果输入某个 `token ID i`，输出就是 `weight[i]` 也就是矩阵的第 i 行。
```
输入：token_ids.shape  == [B, S]          # 每个位置一个整数
输出：embeds.shape     == [B, S, d_model] # 每个位置变成一个 d_model 维向量
```
> 假设 d_model = 4，ID 为 545 的词可能对应向量 `[0.023, -0.451, 0.892, -0.033]`
> 这 4 个数字就是这个词的“身份编码”。训练开始时，这些数字是完全随机的；训练结束时，语义相近的词（如“苹果”和“香蕉”）的向量会在空间中彼此靠近，而语义无关的词（如“苹果”和“民主”）则相距甚远。
### 2.2.1 基础件
#### nn.Module：所有模块的基类
如果想自己实现一个神经网络层，通常都要继承 nn.Module。
它帮你做很多重要的事：
- **自动管理子模块**：如果你在 `__init__` 里定义了其他 `nn.Module`，它会自动追踪，无需手动维护列表。
- **自动追踪参数**：所有被 `nn.Parameter` 包装的张量，都会被它收集到 `.parameters()` 里，供优化器取用。
- **设备迁移一键搞定**：`.to(device)` 会递归地把所有参数和子模块搬到 GPU。
- **模式切换**：`.train()` / `.eval()` 会同步影响所有子模块的 Dropout、BatchNorm 等行为。
- **保存加载**：`torch.save(model.state_dict(), ...)` 和 `model.load_state_dict(...)` 开箱即用。

#### nn.Parameter：告诉 PyTorch“这是可训练参数”
一个普通的 `torch.Tensor` 只是一块内存里的数据。PyTorch 并不知道它是不是模型的参数，也不知道该不该给它算梯度。

`nn.Parameter` 就是对 Tensor 的一个包装声明，相当于对 PyTorch 说：

> “这个张量是模型的权重，**请你务必把它加入可训练参数列表**，计算梯度，并让优化器更新它。”

```python
self.weight = nn.Parameter(torch.empty(vocab_size, d_model))
```
这行代码之后：
- `self.weight` 会出现在 `model.parameters()` 中。
- 反向传播时会计算 `self.weight.grad`。
- 优化器的 `step()` 会更新它的数值。

#### register_buffer：不训练，但要跟着模型走
有些数据虽然不是可训练参数，但却是模型推理和训练所必需的一部分，例如：
- 因果注意力里的 `causal mask`（一个巨大的上三角 `-inf` 矩阵）。
- 位置编码表（如果不使用 RoPE 那种动态计算的方式）。
- 某些统计量缓存。

我们希望这些数据满足三点：
1. **不被优化器更新**（`requires_grad=False`）。
2. **能跟着模型一起搬到 GPU**（`.to(device)` 时自动跟随）。
3. **能跟着模型一起保存和加载**（`state_dict()` 里要有它）。

这时候就用 `register_buffer(name, tensor)`：

```python
self.register_buffer("causal_mask", torch.tensor(...))
```
#### torch.empty：只分配内存，不初始化数值
`torch.empty(shape)` 做的事情只是向操作系统申请一块连续内存，**不往里写任何东西**。里面的值是内存里残留的脏数据。

既然如此，为什么还要用它，而不是用 `torch.zeros` 或 `torch.ones`？

**原因：效率。**
当我们紧接着要手动初始化参数时（比如用正态分布填充），先用 `zeros` 写一遍全零，再覆盖一遍随机数，等于白白多写了一次内存。对于 `vocab_size=152064, d_model=4096` 这种规模的矩阵（约 6 亿个浮点数，占 2.4 GB 显存），少一次内存写入就是可观的加速。
```python
weight = torch.empty(vocab_size, d_model)
nn.init.normal_(weight, mean=0.0, std=1.0)  # 立即覆盖
```
### 2.2.2 实现思路
> 第二步里随机初始化不能随便来。如果初始权重过大，点积计算容易产生极大值，导致 Softmax 后梯度饱和；如果过小，信号可能消失。Embedding 层通常采用 **截断正态分布**（Truncated Normal）。因为正态分布理论上可以产生 `±∞`，尽管概率极低，但在数十亿参数里，偶尔蹦出一个 `±5` 甚至 `±10` 的离群值，就会在训练初期制造数值不稳定。截断是一种廉价的保险。

In [ ]:
import torch
import torch.nn as nn

class MyEmbedding(nn.Module):
    # 第一步：创建参数矩阵
    def __init__(self, vocab_size, d_model):
        super().__init__()
        # 申请内存不初始化
        weight = torch.empty(vocab_size, d_model)
        # 包装成可训练参数让Pytorch追踪
        self.weight = nn.Parameter(weight)
    # 第二步：初始化参数
    def _init_weight(self):
        # 使用截断正态分布初始化
        nn.init.trunc_normal_(
            self.weight,
            mean=0.0,
            std=1.0,
            a=-3.0,     # 上界
            b=3.0       # 下界
        )
    
    # 第三步：前向传播：查表
    def forward(self, token_ids):
        # token_ids: [B, S]
        # 返回: [B, S, d_model]
        return self.weight(token_ids)

> 事实上，PyTorch 官方的 `nn.Embedding` 做的也就是这几件事，只不过额外处理了 `padding_idx`（指定一个 ID 对应的向量永远为 0 且不更新）、`max_norm`（限制向量模长）等边缘功能。

Embedding 层的参数量是 `vocab_size × d_model`。在如今的大模型里，这个数字非常恐怖：

| 模型 | vocab_size | d_model | Embedding 参数量 |
|------|------------|---------|------------------|
| LLaMA-7B | 32,000 | 4096 | 约 1.31 亿 |
| Qwen-72B | 152,064 | 8192 | 约 12.45 亿 |
| DeepSeek-V2 | 102,400 | 5120 | 约 5.24 亿 |

单是词表嵌入就动辄数亿参数，占到模型总参数的 **15%～30%**。因此，几乎所有现代大模型都会采用一个技巧：**Weight Tying（权重绑定）**。

还记得最后一步的 **LM Head** 吗？它也是一个线性层，权重形状为 `[vocab_size, d_model]`（或转置）。既然输入 Embedding 和输出 LM Head 都是要将向量映射到/自词表空间，完全可以让它们 **共享同一份参数矩阵**。

这样做的收益有三重：
1. **大幅降低参数量**：直接省掉一个 `vocab_size × d_model` 的大矩阵。
2. **语义一致性**：输入空间和输出空间使用同一套向量基，逻辑更自洽。
3. **梯度更丰富**：同一个参数既接收来自输入端的梯度，也接收来自输出端的梯度，训练信号更强。

```python
# 输入嵌入
self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)

# 输出头（直接复用 embed_tokens 的权重）
self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
self.lm_head.weight = self.embed_tokens.weight   # 指针绑定，共享内存
# 注意这里是 `=` 直接赋值，不是复制一份，所以两个层指向的是同一块内存，梯度更新自然同步。
```

## 2.3 Linear
如果你把 Transformer 拆开来看，会发现除了 Attention 里的点积和 Softmax，以及一些归一化层和激活函数，剩下的几乎全是 **Linear 层**。
- Q / K / V 投影：三个 Linear（`d_model → d_model`）
- 注意力输出投影：一个 Linear（`d_model → d_model`）
- FFN 升维：一个 Linear（`d_model → d_ff`，通常 `d_ff = 4 × d_model`）
- FFN 降维：一个 Linear（`d_ff → d_model`）
- LM Head：一个 Linear（`d_model → vocab_size`）

一个标准的 LLaMA 风格 Block 里，就有 **7 个 Linear 层**。堆叠 32 层就是 224 个，再加上 Embedding 和 Final Norm，整个模型几乎就是由 Linear 层叠出来的。

### 2.3.1 数学定义与工程实现
在大学的线性代数课上，线性变换通常写成 $$ \mathbf{y} = W \mathbf{x} $$
其中：
- $W \in \mathbb{R}^{\text{out} \times \text{in}}$（输出维度 × 输入维度）
- $\mathbf{x} \in \mathbb{R}^{\text{in} \times 1}$（列向量）
- $\mathbf{y} \in \mathbb{R}^{\text{out} \times 1}$

但在深度学习框架（PyTorch / JAX / TensorFlow）的实际代码中，输入通常不是单个列向量，而是一个 **batch 的行向量堆叠**：
```python
x.shape == [batch_size, in_features]
```
如果直接写 `W @ x`，维度就对不上了：`[out, in] @ [batch, in]` 会报错。

因此，框架的 Linear 层通常存储权重为 `[out_features, in_features]`，但**实际计算时做转置乘法**：

```python
y = x @ weight.T   # [batch, in] @ [in, out] = [batch, out]
```

或者等价地用 `F.linear(x, weight, bias)`，其内部逻辑也是：

```python
y = torch.matmul(x, weight.t())
```

### 2.3.2 为什么权重通常存成 `[out, in]`？
**1. 语义清晰：一行就是一个神经元的全部输入权重**

权重矩阵的第 $i$ 行，就是输出第 $i$ 个神经元与所有输入维度的连接强度：

```
         in_0   in_1   in_2   ...   in_{d_in-1}
out_0: [  w00    w01    w02    ...      w0n    ]
out_1: [  w10    w11    w12    ...      w1n    ]
...
```

你取出 `weight[0]`，就得到了“产生第一个输出维度所需要的全部权重向量”。在做参数可视化、剪枝、分析时，这种组织方式最符合直觉。

**2. 与参数初始化和正则化的习惯兼容**

Xavier / Kaiming 初始化公式中的 `fan_in` 和 `fan_out`，正是基于这种形状定义的：

- `fan_in = in_features`（每一行有多少个输入连接）
- `fan_out = out_features`（每一列有多少个输出连接）

**3. 转置乘法在 GPU 上高度优化**

`x @ weight.T` 这种形式的矩阵乘法是 BLAS 库的标准操作（`GEMM`），cuBLAS 对其有极致优化。反过来，如果存成 `[in, out]`，计算 `x @ weight` 也一样快，但权重矩阵的“行”就不再对应单个输出神经元了。权衡之下，`[out, in]` 更符合开发者的心智模型。

### 2.3.3 初始化中的方差控制
Linear 层如果初始化不当，会引起两种灾难：
- **梯度爆炸**：层间传递的数值越来越大，最终变成 `NaN`。
- **梯度消失**：反向传播时梯度越来越小，模型学不动。
为了防止这些情况，我们需要**控制初始权重的大小**，让信号在前向和反向传播中保持稳定。

#### Xavier 初始化思想

Xavier Glorot 等人在 2010 年提出了一个经典原则：**让每一层输出的方差等于输入的方差**。

对于 Linear 层 $y = x W^\top$（忽略激活函数），假设：
- 输入 $x$ 的每个元素方差为 $\sigma_x^2$
- 权重 $W$ 的每个元素方差为 $\sigma_w^2$

那么输出 $y$ 的每个元素的方差近似为：

$$ \text{Var}(y_i) = \text{fan\_in} \cdot \sigma_x^2 \cdot \sigma_w^2 $$

为了保持方差不变，即 $\text{Var}(y_i) = \sigma_x^2$，我们需要：

$$ \sigma_w^2 = \frac{1}{\text{fan\_in}} $$

这就是 **Xavier 均匀/正态初始化**的来历。后来考虑到反向传播时梯度的方差，又出现了折衷版本：

$$ \sigma_w^2 = \frac{2}{\text{fan\_in} + \text{fan\_out}} $$
#### 实践中用的截断正态

在 Transformer 的具体实现中，Linear 层常采用**截断正态分布**（Truncated Normal）进行初始化，标准差为：

$$ \sigma = \sqrt{\frac{2}{\text{fan\_in} + \text{fan\_out}}} $$

例如 LLaMA 的官方代码中，Linear 权重初始化类似：

```python
std = (2 / (in_features + out_features)) ** 0.5
nn.init.trunc_normal_(weight, mean=0.0, std=std, a=-3*std, b=3*std)
```

- `mean=0.0`：对称分布，避免引入偏置。
- 截断范围 `[-3σ, 3σ]`：切掉极少量离群值，防止训练初期出现异常大值。

#### Bias 初始化

Linear 层通常带有可选的偏置项 `bias`。如果启用，它一般被初始化为 **全零**。因为权重已经对称了，不需要额外的偏置来打破对称性（不像 ReLU 有时需要小的正偏置来避免神经元死亡）。

在 Transformer 中，为了简化训练动态，**很多实现（包括 LLaMA、Qwen）的 Q/K/V/O 投影和 FFN 都直接关掉 bias**。只有某些特定层（如 LM Head）可能保留 bias。

```python
self.q_proj = nn.Linear(d_model, d_model, bias=False)
```

### 2.3.4 `torch.einsum`
最普通的 Linear 前向可以用一行代码写完：

```python
def forward(self, x):
    return x @ self.weight.T + self.bias
```
但如果我们维护的是一个 Transformer 代码库，你可能会遇到各种形状的输入：
- 单 batch：`[B, d_model]`
- 序列 batch：`[B, S, d_model]`
- 多头注意力中间状态：`[B, num_heads, S, d_head]`

直接写 `x @ weight.T` 在这些多维情况下依然能工作（因为 `@` 支持 broadcasting），但**可读性**会下降——需要在心里默默对齐矩阵乘法的最后两维，还要时刻注意 `bias` 的广播是否正确。

torch.einsum` 提供了一种基于爱因斯坦求和约定的显式写法，把维度关系**写在脸上**：

```python
y = torch.einsum('...i,oi->...o', x, self.weight)
```

- `x` 的形状标记为 `...i`：
  - `...` 代表任意数量的前置维度（如 `B`、`S`、`num_heads` 等），原样保留。
  - `i` 代表最后一个维度，即 `in_features`。
- `self.weight` 的形状标记为 `oi`：
  - `o` 代表输出维度 `out_features`。
  - `i` 代表输入维度 `in_features`。
- `->...o` 表示：
  - 对下标 `i` 求和（因为它在输入端出现了但在输出端消失了）。
  - 保留所有的前置维度 `...` 并加上输出维度 `o`。

效果等价于 `x @ weight.T`，但**更通用、更自文档化**：

- 无论 `x` 是 `[B, d]` 还是 `[B, S, d]` 还是 `[B, H, S, d]`，这一行代码通通搞定。
- 读代码的人一眼就能看到“输入维 i 被求和消掉了，输出变成了 o”。

注意 `einsum` 里的 `self.weight` 下标是 `oi`，这正好匹配我们存储的形状 `[out, in]`。我们不需要手动 `.T`，因为 `einsum` 会自动根据下标对齐维度。这避免了因忘记转置而引发的隐性 bug。
### 2.3.5 实现思路

In [ ]:
import torch
import torch.nn as nn
import math

class TransformerLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=False):
        super().__init__()
        # 1. 创建权重矩阵 [out, in]
        weight = torch.empty(out_features, in_features)
        self.weight = nn.Parameter(weight)

        # 2. 可选的偏置
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_buffer('bias', None)

        # 3. 初始化权重
        self._init_weight()
    def _init_weight(self):
        # Xavier风格标准差
        fan_in, fan_out = self.weight.shape[1], self.weight.shape[0]
        std = math.sqrt(2.0 / (fan_in + fan_out))
        nn.init.trunc_normal_(self.weight, mean=0.0, std=std, a=-3.0, b=3.0)
    
    def forward(self, x):
        # x: [..., in_features]
        y = torch.einsum('...i,oi->...o', x, self.weight)
        if self.bias is not None:
            y = y + self.bias
        return y



In [ ]:
# 模拟一个 Q 投影
# q_proj = TransformerLinear(in_features=4096, out_features=4096, bias=False)
# hidden = torch.randn(2, 128, 4096)  # [Batch, SeqLen, d_model]
# q = q_proj(hidden)  
# print("输出形状:", q.shape)  # 正确输出: torch.Size([2, 128, 4096])


输出形状: torch.Size([2, 128, 4096])
